In [ ]:
import camb
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import pysm3
import pysm3.units as u

from pathlib import Path
from astropy.table import Table, vstack

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

mpl.rcParams['font.family']       = 'serif'
mpl.rcParams['font.serif']        = ['DejaVu Serif', 'Times New Roman', 'serif']
mpl.rcParams['mathtext.fontset']  = 'dejavuserif'   
mpl.rcParams['text.usetex']       = False            

mpl.rcParams['font.size']         = 14
mpl.rcParams['axes.titlesize']    = 15
mpl.rcParams['axes.labelsize']    = 14
mpl.rcParams['xtick.labelsize']   = 12
mpl.rcParams['ytick.labelsize']   = 12
mpl.rcParams['legend.fontsize']   = 12
mpl.rcParams['legend.title_fontsize'] = 13


mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['savefig.dpi']       = 600         
mpl.rcParams['savefig.bbox']      = 'tight'
mpl.rcParams['savefig.format']    = 'pdf'       

mpl.rcParams['axes.spines.top']   = True
mpl.rcParams['axes.spines.right'] = True
mpl.rcParams['axes.edgecolor']    = 'black'
mpl.rcParams['axes.linewidth']    = 1.2  


N_colores = 8                                      
cmap      = mpl.colormaps['rainbow']               
colores   = [cmap(i / (N_colores - 1)) for i in range(N_colores)]
mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=colores)

# --- Grilla ---
mpl.rcParams['axes.grid']         = True
mpl.rcParams['grid.color']        = '0.85'
mpl.rcParams['grid.linestyle']    = '--'
mpl.rcParams['grid.linewidth']    = 0.6
mpl.rcParams['grid.alpha']        = 0.7

# --- Ticks ---
mpl.rcParams['xtick.direction']   = 'in'
mpl.rcParams['ytick.direction']   = 'in'
mpl.rcParams['xtick.major.size']  = 5
mpl.rcParams['ytick.major.size']  = 5
mpl.rcParams['xtick.minor.size']  = 3
mpl.rcParams['ytick.minor.size']  = 3
mpl.rcParams['xtick.major.width'] = 1.0
mpl.rcParams['ytick.major.width'] = 1.0
mpl.rcParams['xtick.minor.visible'] = True
mpl.rcParams['ytick.minor.visible'] = True

# --- Lineas y markers ---
mpl.rcParams['lines.linewidth']   = 2.0
mpl.rcParams['lines.markersize']  = 6

# --- Leyenda ---
mpl.rcParams['legend.frameon']    = True
mpl.rcParams['legend.framealpha'] = 0.85
mpl.rcParams['legend.edgecolor']  = '0.8'
mpl.rcParams['legend.fancybox']   = False

mpl.rcParams['image.cmap']        = 'rainbow'    

In [ ]:
pars = camb.set_params(H0=67.5, ombh2=0.022, omch2=0.122, mnu=0.06, omk=0, tau=0.06, As=2e-9, ns=0.965, lmax=1500)

results = camb.get_results(pars)
powers = results.get_cmb_power_spectra(pars, CMB_unit="muK", raw_cl=True) 
cl_tt = powers["lensed_scalar"][:, 0]
ells = np.arange(len(cl_tt))
cl_tt[:2] = 0.0
Dl_tt = ells * (ells + 1) * cl_tt / (2 * np.pi)

plt.figure(figsize=(8,5))
plt.plot(ells[2:], cl_tt[2:], color='pink')
plt.yscale("log")
plt.xlabel(r"$\ell$")
plt.ylabel(r"$C_\ell^{TT}\,[\mu \mathrm{K}^2]$")
plt.show()

plt.figure(figsize=(8,5))
plt.plot(ells[2:], Dl_tt[2:], color='pink')
plt.xlabel(r"$\ell$")
plt.ylabel(r"$D_\ell^{TT}\,[\mu \mathrm{K}^2]$")
plt.show()

In [ ]:
ells

In [ ]:
seed = 113403063
np.random.seed(seed)

lmax_master = len(cl_tt) - 1
alm_master = hp.synalm(cl_tt, lmax=lmax_master, new=True, verbose=False)
NSIDE = 1024
lmax1 = 3*NSIDE - 1
lmax_map = min(lmax_master, lmax1)
alm = hp.resize_alm(alm_master, lmax=lmax_master, mmax=lmax_master, lmax_out=lmax_map, mmax_out=lmax_map)
map_1024 = hp.alm2map(alm, nside=NSIDE, lmax=lmax_map, pol=False, verbose=False)


hp.mollview(map_1024, title=f"NSIDE = {NSIDE}", cmap="coolwarm")
hp.graticule()
plt.show()

In [ ]:
print(f"Número de pixeles en este mapa (NSIDE = {NSIDE}):", len(map_1024))
pixeles=12*NSIDE*NSIDE
print(f"pixeles = {pixeles}")

In [ ]:
temp_promedio = np.mean(map_1024)
temp_desv = np.std(map_1024)
print(f"Temperatura promedio: {temp_promedio:.12f} microK")

In [ ]:
seed = 207790273
rng = np.random.default_rng(seed)
npix = 12*NSIDE*NSIDE
pix_elegido = rng.integers(0, npix)
theta_pix, phi_pix = hp.pix2ang(NSIDE, pix_elegido)
vec0 = hp.ang2vec(theta_pix, phi_pix)
temp0 = map_1024[pix_elegido]

print("Pixel elegido:", pix_elegido)
print("theta [rad]:", theta_pix)
print("phi [rad]:", phi_pix)
print("Temperatura en ese punto [microK]:", temp0)

In [ ]:
hp.mollview(map_1024, title=f"NSIDE = {NSIDE}", cmap="coolwarm")
hp.projscatter(theta_pix, phi_pix, marker="o", s=60, color='green')
hp.graticule()
plt.show()

In [ ]:
NSIDE=1024
seed = 113403063
rng = np.random.default_rng(seed)
n_puntos = 20860
npix = 12*NSIDE*NSIDE
r_inicial = 0.2
r_final = 30.2
ancho_anillo = 0.5
j=0
bordes_radiales = np.arange(r_inicial, r_final + ancho_anillo, ancho_anillo)

radios_internos = bordes_radiales[:-1] #eliminamos el último radio
radios_externos = bordes_radiales[1:] #eliminamos el primer radio
radios_medios = (radios_internos + radios_externos) / 2

temperaturas_todos_los_puntos = [] #Acá juntamos las temperaturas de todos los puntos contenidos en un anillo para cada punto. Será una lista de listas
pixeles_elegidos = [] #Acá se guardan los pixeles aleatorios con los que se trabajarán (su longitud corresponde al número de puntos aleatorios)
temps_puntos = [] #Acá se guardarán las temperaturas medias de cada anillo
temperatura_media_anillos = [] #la longitud  de esta lista debiese ser equivalente al número de puntos, cada elemento es una lista
for i in range(n_puntos):
    j+=1
    print(j)
    pix_elegido = rng.integers(0, npix) #para cada punto se elige un pixel aleatorio
    pixeles_elegidos.append(pix_elegido)
    theta_pix, phi_pix = hp.pix2ang(NSIDE, pix_elegido) #pixel a coordenadas angulares
    vec_i = hp.ang2vec(theta_pix, phi_pix) #dirección angular a un vector 3d
    temperatura_media_anillos_i = []
    for r_interno_anillo, r_externo_anillo in zip(radios_internos, radios_externos):
        pix_disco_externo = hp.query_disc(nside=NSIDE, vec=vec_i, radius=np.radians(r_externo_anillo), inclusive=True)
        pix_disco_interno = hp.query_disc(nside=NSIDE, vec=vec_i, radius=np.radians(r_interno_anillo), inclusive=True)
        pix_anillo = np.setdiff1d(pix_disco_externo, pix_disco_interno)
        temp_media = np.mean(map_1024[pix_anillo])
        temperatura_media_anillos_i.append(temp_media)
    temperatura_media_anillos.append(temperatura_media_anillos_i)


In [ ]:
len(temperatura_media_anillos)

In [ ]:
a = np.array(temperatura_media_anillos)
temperatura_promedios = np.mean(a, axis=0)


plt.figure(figsize=(8, 5))
plt.plot(radios_medios, temperatura_promedios, marker="o", linestyle="-", color="#631D51")

plt.xlabel("Radio medio anillo [deg]")
plt.ylabel("Temperatura media [microK]")

plt.grid(True)
plt.show()

In [ ]:
theta_pix, phi_pix = hp.pix2ang(NSIDE, pixeles_elegidos)

hp.mollview(
    map_1024, title=f"Puntos aleatorios sobre mapa CMB | NSIDE = {NSIDE}", cmap="coolwarm", unit="microK"
)

hp.projscatter(theta_pix, phi_pix, marker=".", s=2, color="green", alpha=0.5
)

hp.graticule()
plt.show()

# 2MASS

In [ ]:
ROOT = Path.home() / "Descargas" / "2mrs_v240"

done_matches = list(ROOT.rglob("2mrs_1175_done.fits"))
nocz_matches = list(ROOT.rglob("2mrs_1175_nocz.fits"))

print("Archivos done encontrados:", done_matches)
print("Archivos nocz encontrados:", nocz_matches)

done_file = done_matches[0]
nocz_file = nocz_matches[0]

cat_done = Table.read(done_file)
cat_nocz = Table.read(nocz_file)

cat = vstack([cat_done, cat_nocz], join_type="outer", metadata_conflicts="silent")

print("Total de galaxias cargadas:", len(cat))
print(cat.colnames)

In [ ]:
def col_float(tabla, nombre):
    col = tabla[nombre]
    if hasattr(col, "filled"):
        col = col.filled(np.nan)
    return np.array(col, dtype=float)

k_c = col_float(cat, "MKC")    
b_gal = col_float(cat, "GLAT") 

mask_submuestra = (np.isfinite(k_c) & np.isfinite(b_gal) & (k_c <= 11.254) & (np.abs(b_gal) > 10))
sub = cat[mask_submuestra]
print("Galaxias en la submuestra:", len(sub))

In [ ]:
def col_float(tabla, nombre):
    col = tabla[nombre]
    if hasattr(col, "filled"):
        col = col.filled(np.nan)
    return np.array(col, dtype=float)

k_c = col_float(cat, "MKC")    
b_gal = col_float(cat, "GLAT") 

mask_submuestra = (np.isfinite(k_c) & np.isfinite(b_gal) & (k_c <= 11.254) & (np.abs(b_gal) > 10))
sub = cat[mask_submuestra]
print("Galaxias en la submuestra:", len(sub))

In [ ]:
from tqdm.auto import tqdm

NSIDE = 1024

n_puntos = len(sub)   # ahora el número de puntos es el número de galaxias

r_inicial = 0.2
r_final = 30.2
ancho_anillo = 0.5

bordes_radiales = np.arange(r_inicial, r_final + ancho_anillo, ancho_anillo)

radios_internos = bordes_radiales[:-1]
radios_externos = bordes_radiales[1:]
radios_medios = (radios_internos + radios_externos) / 2

l_gal = col_float(sub, "GLON")   # longitud galáctica en grados
b_gal = col_float(sub, "GLAT")   # latitud galáctica en grados

theta_gal = np.radians(90.0 - b_gal)
phi_gal = np.radians(l_gal)

pixeles_galaxias = hp.ang2pix(NSIDE, theta_gal, phi_gal)

temperatura_media_anillos = []
pixeles_elegidos = []

for i in tqdm(range(n_puntos)):

    pix_galaxia = pixeles_galaxias[i]
    pixeles_elegidos.append(pix_galaxia)
    vec_i = hp.ang2vec(theta_gal[i], phi_gal[i])
    temperatura_media_anillos_i = []

    for r_interno_anillo, r_externo_anillo in zip(radios_internos, radios_externos):

        pix_disco_externo = hp.query_disc(nside=NSIDE, vec=vec_i, radius=np.radians(r_externo_anillo), inclusive=True)
        pix_disco_interno = hp.query_disc(nside=NSIDE, vec=vec_i, radius=np.radians(r_interno_anillo), inclusive=True)
        pix_anillo = np.setdiff1d(pix_disco_externo, pix_disco_interno )
        temp_media = np.mean(map_1024[pix_anillo])
        temperatura_media_anillos_i.append(temp_media)
    temperatura_media_anillos.append(temperatura_media_anillos_i)

temperatura_media_anillos = np.array(temperatura_media_anillos)
pixeles_elegidos = np.array(pixeles_elegidos)

print("Shape temperatura_media_anillos:", temperatura_media_anillos.shape)
print("Shape pixeles_elegidos:", pixeles_elegidos.shape)

In [ ]:
temperatura_promedios_2mrs = np.mean(temperatura_media_anillos, axis=0)

plt.figure(figsize=(8, 5))

plt.plot(radios_medios, temperatura_promedios_2mrs, marker="o", linestyle="-", color="#631D51")
plt.axhline(0, linestyle="-", linewidth=1)

plt.xlabel("Radio medio anillo [deg]")
plt.ylabel("Temperatura media [microK]")

plt.grid(True)
plt.tight_layout()
plt.show()